Groundwater | Flow Modeling Track

# Topic 6: Sensitivity and Uncertainty Analysis


**Step 7 of the Groundwater Modelling Process**

In this notebook, we explore how model predictions respond to parameter changes (sensitivity) and how to quantify the uncertainty in our predictions. These analyses are essential for understanding model reliability and guiding data collection efforts.

---

### Learning Objectives

By the end of this notebook, you will be able to:

1. Distinguish between **sensitivity analysis** and **uncertainty analysis**
2. Perform simple **one-at-a-time (OAT)** sensitivity analysis
3. Interpret sensitivity results using **tornado diagrams**
4. Understand sources of **uncertainty** in groundwater models
5. Recognize when **advanced methods** are needed

---

### Notebook Structure

1. Introduction: Why Sensitivity and Uncertainty Matter
2. Sensitivity Analysis Concepts
3. Simple Illustrative Example: OAT Sensitivity
4. Uncertainty Analysis Concepts
5. Linking Sensitivity to Uncertainty
6. Advanced Topics
7. Discussion
8. References

---

> **Link to Your Case Study**: In your group case study notebooks (e.g., `case_study_transport_group_0.ipynb`, Section 15), you will apply sensitivity analysis to transport parameters. This notebook provides the conceptual foundation for that work.

In [ ]:
# Import libraries
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

import flopy
from flopy.utils import HeadFile

from IPython.display import display

# Load local modules
sys.path.append('../../_SUPPORT/src')
sys.path.append('../../_SUPPORT/src/scripts/scripts_exercises')

from data_utils import (
    download_named_file,
    get_default_data_folder,
)

# Set up plotting defaults
plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['font.size'] = 12

## 1 - Introduction: Why Sensitivity and Uncertainty Matter

### The Fundamental Questions

After calibrating and validating a model (to the extent possible), two critical questions remain:

1. **Sensitivity**: Which parameters most strongly influence model predictions?
2. **Uncertainty**: How confident can we be in our predictions?

### Sensitivity vs. Uncertainty

| Aspect | Sensitivity Analysis | Uncertainty Analysis |
|--------|---------------------|---------------------|
| **Question** | How much do outputs change when inputs change? | What is the range of possible outputs? |
| **Focus** | Parameter importance ranking | Prediction confidence intervals |
| **Input** | Systematic parameter variations | Parameter probability distributions |
| **Output** | Sensitivity indices, tornado diagrams | Probability distributions, confidence bounds |

### When to Perform These Analyses

The timing depends on the project context:

| Timing | Rationale |
|--------|----------|
| **Before calibration** | Identify which parameters can be estimated from available data |
| **After calibration** | Assess prediction reliability, guide additional data collection |
| **Before application** | Understand which uncertainties matter for the decision at hand |
| **Iteratively** | Refine understanding as model develops |

### Connection to Validation (Notebook 6)

Recall from Notebook 6 that our model has **limited validation** due to insufficient observation data. Sensitivity and uncertainty analysis help us:

- Understand which parameters drive predictions in unobserved areas
- Identify where additional data would most reduce uncertainty
- Provide appropriate caveats on model predictions

## 2 - Sensitivity Analysis Concepts

### Types of Sensitivity Analysis

#### Local Sensitivity (One-at-a-Time, OAT)

The simplest approach: vary one parameter at a time while holding others constant.

**Process**:
1. Define a baseline parameter set
2. Perturb one parameter (e.g., ±10%, ±20%)
3. Run the model and record output change
4. Repeat for each parameter

**Advantages**:
- Simple to implement and interpret
- Low computational cost
- Clear cause-effect relationships

**Limitations**:
- Ignores parameter interactions
- Results depend on baseline values
- May miss important sensitivities in nonlinear systems

#### Global Sensitivity Analysis

Varies all parameters simultaneously across their full ranges.

**Methods** (covered in Advanced Topics):
- Morris screening method
- Sobol variance-based indices
- Monte Carlo-based approaches

### Which Parameters to Test?

For groundwater flow models, key parameters include:

| Parameter | Symbol | Typical Range | Notes |
|-----------|--------|---------------|-------|
| Hydraulic conductivity | K | Orders of magnitude | Often most sensitive |
| Recharge | R | ±50% or more | Climate-dependent |
| Porosity | n | ±20% | Affects transport more than flow |
| Storage coefficient | S | Orders of magnitude | Transient models only |
| River conductance | C | ±50% | If river boundaries present |
| Boundary heads | h | ±1-5 m | If head boundaries present |

### Interpreting Sensitivity Results

**Normalized Sensitivity**:

To compare sensitivities across parameters with different units:

$$S_i = \frac{\Delta O / O}{\Delta P_i / P_i}$$

Where:
- $S_i$ = normalized sensitivity to parameter $i$
- $\Delta O$ = change in output
- $O$ = baseline output
- $\Delta P_i$ = change in parameter $i$
- $P_i$ = baseline parameter value

A sensitivity of 1.0 means a 10% parameter change causes a 10% output change.

## 3 - Simple Illustrative Example: OAT Sensitivity

Let's perform a simple one-at-a-time sensitivity analysis on our Limmat Valley model.

In [ ]:
# Load the calibrated model
model_name = 'limmat_valley_model_nwt'
workspace = os.path.join(get_default_data_folder(), 'calibration')

if not os.path.exists(workspace):
    print(f"ERROR: Model workspace not found at: {workspace}")
    print("Please run notebooks 4 and 5 first.")
else:
    print(f"Model workspace: {workspace}")

In [ ]:
# Load the model
mf = flopy.modflow.Modflow.load(
    f"{model_name}.nam",
    model_ws=workspace,
    exe_name='mfnwt',
    version='mfnwt',
    verbose=False
)

print(f"Model loaded: {mf.name}")
print(f"Grid: {mf.nlay} layer(s), {mf.nrow} rows, {mf.ncol} columns")

In [ ]:
# Get baseline parameter values
print("="*60)
print("BASELINE PARAMETER VALUES")
print("="*60)

# Hydraulic conductivity
if hasattr(mf, 'upw'):
    hk_baseline = mf.upw.hk.array.copy()
    print(f"\nHydraulic Conductivity (K):")
    print(f"  Mean: {np.mean(hk_baseline[hk_baseline > 0]):.2e} m/day")
elif hasattr(mf, 'lpf'):
    hk_baseline = mf.lpf.hk.array.copy()
    print(f"\nHydraulic Conductivity (K):")
    print(f"  Mean: {np.mean(hk_baseline[hk_baseline > 0]):.2e} m/day")

# Recharge
if hasattr(mf, 'rch') and mf.rch is not None:
    rch_baseline = mf.rch.rech.array.copy()
    # Handle different array dimensions
    if rch_baseline.ndim == 4:
        rch_2d = rch_baseline[0, 0]
    elif rch_baseline.ndim == 3:
        rch_2d = rch_baseline[0]
    else:
        rch_2d = rch_baseline
    rch_mean = np.mean(rch_2d[rch_2d > 0])
    print(f"\nRecharge:")
    print(f"  Mean: {rch_mean:.2e} m/day ({rch_mean * 365 * 1000:.1f} mm/year)")

In [ ]:
# Define sensitivity analysis function
def run_sensitivity_trial(mf, k_factor=1.0, rch_factor=1.0, workspace=None):
    """
    Run model with modified parameters and return mean head.
    
    Parameters
    ----------
    mf : flopy.modflow.Modflow
        The MODFLOW model object
    k_factor : float
        Multiplier for hydraulic conductivity (1.0 = no change)
    rch_factor : float
        Multiplier for recharge (1.0 = no change)
    workspace : str
        Model workspace directory
        
    Returns
    -------
    float
        Mean head in active cells, or None if model fails
    """
    import copy
    import tempfile
    import shutil
    
    # Create temporary workspace
    temp_ws = tempfile.mkdtemp()
    
    try:
        # Copy model to temp workspace
        for f in os.listdir(workspace):
            src = os.path.join(workspace, f)
            if os.path.isfile(src):
                shutil.copy2(src, temp_ws)
        
        # Load fresh model in temp workspace
        mf_temp = flopy.modflow.Modflow.load(
            f"{mf.name}.nam",
            model_ws=temp_ws,
            exe_name='mfnwt',
            version='mfnwt',
            verbose=False
        )
        
        # Modify K
        if k_factor != 1.0:
            if hasattr(mf_temp, 'upw'):
                mf_temp.upw.hk = mf_temp.upw.hk.array * k_factor
            elif hasattr(mf_temp, 'lpf'):
                mf_temp.lpf.hk = mf_temp.lpf.hk.array * k_factor
        
        # Modify recharge - handle different array dimensions
        if rch_factor != 1.0 and hasattr(mf_temp, 'rch') and mf_temp.rch is not None:
            rch_array = mf_temp.rch.rech.array.copy()
            # Get the 2D array regardless of original dimensions
            if rch_array.ndim == 4:
                # Shape is (nper, 1, nrow, ncol) - extract 2D and apply factor
                rch_2d = rch_array[0, 0] * rch_factor
                # Create stress period data dictionary with 2D arrays
                rch_spd = {0: rch_2d}
            elif rch_array.ndim == 3:
                rch_2d = rch_array[0] * rch_factor
                rch_spd = {0: rch_2d}
            else:
                rch_2d = rch_array * rch_factor
                rch_spd = {0: rch_2d}
            mf_temp.rch.rech = rch_spd
        
        # Write and run
        mf_temp.write_input()
        success, _ = mf_temp.run_model(silent=True)
        
        if success:
            # Read heads
            hds = HeadFile(os.path.join(temp_ws, f"{mf.name}.hds"))
            head = hds.get_data(kstpkper=(0, 0))
            
            # Calculate mean head in active cells
            head_valid = head[head > -999]
            return np.mean(head_valid)
        else:
            return None
            
    finally:
        # Clean up
        shutil.rmtree(temp_ws, ignore_errors=True)

In [ ]:
# Run OAT sensitivity analysis
print("="*60)
print("ONE-AT-A-TIME SENSITIVITY ANALYSIS")
print("="*60)

# Define perturbations
perturbations = [-0.2, -0.1, 0.0, 0.1, 0.2]  # -20%, -10%, baseline, +10%, +20%

# Run baseline first
print("\nRunning baseline model...")
baseline_head = run_sensitivity_trial(mf, k_factor=1.0, rch_factor=1.0, workspace=workspace)
print(f"Baseline mean head: {baseline_head:.2f} m")

# Store results
sensitivity_results = {
    'K': {'perturbation': [], 'head': []},
    'Recharge': {'perturbation': [], 'head': []}
}

# Test K sensitivity
print("\nTesting K sensitivity...")
for p in perturbations:
    factor = 1.0 + p
    head = run_sensitivity_trial(mf, k_factor=factor, rch_factor=1.0, workspace=workspace)
    if head is not None:
        sensitivity_results['K']['perturbation'].append(p * 100)
        sensitivity_results['K']['head'].append(head)
        print(f"  K × {factor:.1f}: mean head = {head:.2f} m")

# Test Recharge sensitivity
print("\nTesting Recharge sensitivity...")
for p in perturbations:
    factor = 1.0 + p
    head = run_sensitivity_trial(mf, k_factor=1.0, rch_factor=factor, workspace=workspace)
    if head is not None:
        sensitivity_results['Recharge']['perturbation'].append(p * 100)
        sensitivity_results['Recharge']['head'].append(head)
        print(f"  Rch × {factor:.1f}: mean head = {head:.2f} m")

In [ ]:
# Plot sensitivity results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Line plot showing head vs parameter change
ax = axes[0]
for param, data in sensitivity_results.items():
    if len(data['perturbation']) > 0:
        ax.plot(data['perturbation'], data['head'], 'o-', label=param, linewidth=2, markersize=8)

ax.axhline(y=baseline_head, color='gray', linestyle='--', label='Baseline')
ax.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Parameter Change (%)')
ax.set_ylabel('Mean Head (m a.s.l.)')
ax.set_title('Sensitivity of Mean Head to Parameter Changes')
ax.legend()
ax.grid(alpha=0.3)

# Right: Tornado diagram
ax = axes[1]

# Calculate head change for ±20%
tornado_data = []
for param, data in sensitivity_results.items():
    if len(data['perturbation']) >= 5:
        head_minus20 = data['head'][0]  # -20%
        head_plus20 = data['head'][-1]  # +20%
        
        change_minus = head_minus20 - baseline_head
        change_plus = head_plus20 - baseline_head
        
        tornado_data.append({
            'param': param,
            'low': min(change_minus, change_plus),
            'high': max(change_minus, change_plus),
            'range': abs(change_plus - change_minus)
        })

# Sort by range (most sensitive first)
tornado_data.sort(key=lambda x: x['range'], reverse=True)

# Plot tornado
y_pos = np.arange(len(tornado_data))
for i, d in enumerate(tornado_data):
    ax.barh(i, d['high'] - d['low'], left=d['low'], height=0.6, 
            color='steelblue', alpha=0.7, edgecolor='black')

ax.axvline(x=0, color='black', linewidth=1)
ax.set_yticks(y_pos)
ax.set_yticklabels([d['param'] for d in tornado_data])
ax.set_xlabel('Change in Mean Head (m)')
ax.set_title('Tornado Diagram: Parameter Sensitivity (±20%)')
ax.grid(axis='x', alpha=0.3)

plt.suptitle('Figure 1: One-at-a-Time Sensitivity Analysis Results', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate normalized sensitivities
print("="*60)
print("NORMALIZED SENSITIVITY COEFFICIENTS")
print("="*60)
print("\n(Sensitivity = % change in output / % change in input)")
print("A value of 1.0 means output changes proportionally with input.\n")

sensitivity_table = []

for param, data in sensitivity_results.items():
    if len(data['perturbation']) >= 5:
        # Use +20% perturbation for calculation
        head_change = (data['head'][-1] - baseline_head) / baseline_head * 100
        param_change = 20  # %
        
        sensitivity = head_change / param_change
        
        sensitivity_table.append({
            'Parameter': param,
            'Baseline Head (m)': f"{baseline_head:.2f}",
            'Head at +20% (m)': f"{data['head'][-1]:.2f}",
            'Head Change (%)': f"{head_change:.2f}",
            'Normalized Sensitivity': f"{sensitivity:.3f}"
        })

sensitivity_df = pd.DataFrame(sensitivity_table)
display(sensitivity_df)

print("\nInterpretation:")
print("- Higher absolute values = more sensitive parameters")
print("- Positive values = output increases when parameter increases")
print("- Negative values = output decreases when parameter increases")

### 3.1 - Interpreting the Results

From the sensitivity analysis above, we can observe:

1. **Relative Sensitivity**: The tornado diagram shows which parameters have the largest effect on mean head

2. **Direction of Effect**:
   - Increasing K typically **decreases** heads (water drains faster)
   - Increasing recharge typically **increases** heads (more water input)

3. **Linearity**: If the sensitivity curves are approximately linear, the system behaves predictably. Strong nonlinearity would indicate more complex parameter interactions.

> **Note**: This analysis uses mean head as the output metric. In practice, you would analyze sensitivity at specific locations of interest (e.g., observation wells, prediction points).

## 4 - Uncertainty Analysis Concepts

### Sources of Uncertainty

Uncertainty in groundwater models comes from multiple sources:

| Source | Description | Example |
|--------|-------------|----------|
| **Parameter uncertainty** | Imperfect knowledge of model parameters | K varies spatially, but we use a single value |
| **Structural uncertainty** | Model simplifications may not capture reality | Using 1 layer when aquifer is heterogeneous |
| **Measurement uncertainty** | Errors in observed data | Water level measurement accuracy ±0.1 m |
| **Boundary condition uncertainty** | Unknown or variable boundary conditions | Future recharge under climate change |
| **Scenario uncertainty** | Unknown future conditions | Future pumping rates |

### Uncertainty Propagation

How does input uncertainty translate to output uncertainty?

**Simple Example**:
- If K is uncertain by ±50%
- And head is sensitive to K (sensitivity = 0.5)
- Then head uncertainty ≈ 0.5 × 50% = ±25%

This simple multiplication only works for:
- Linear systems
- Small perturbations
- Single parameters

For real models, we need more sophisticated approaches.

### Representing Uncertainty

**Parameter distributions** describe our knowledge about parameter values:

| Distribution | When to Use |
|--------------|-------------|
| **Uniform** | Only bounds known, all values equally likely |
| **Normal** | Central value known with symmetric uncertainty |
| **Log-normal** | Positive parameters that vary over orders of magnitude (e.g., K) |
| **Triangular** | Minimum, maximum, and most likely value known |

### Confidence Intervals on Predictions

The goal is to express predictions as ranges:

- "Head at well X is predicted to be **402 ± 2 m** (95% confidence)"
- "Contaminant will reach the river in **5-15 years** (90% confidence)"

These ranges acknowledge that our predictions are uncertain, which is essential for honest communication with decision-makers.

## 5 - Linking Sensitivity to Uncertainty

### The Key Insight

**Sensitive parameters with high uncertainty should be prioritized for data collection.**

Consider two parameters:
- Parameter A: High sensitivity, low uncertainty → Prediction is well-constrained
- Parameter B: High sensitivity, high uncertainty → Major source of prediction uncertainty
- Parameter C: Low sensitivity, high uncertainty → Doesn't matter much

### Data Collection Priorities

```
                        Sensitivity
                    Low         High
                ┌───────────┬───────────┐
         High   │  Low      │  HIGH     │  ← Focus data
Uncertainty     │  Priority │  PRIORITY │    collection here
                ├───────────┼───────────┤
         Low    │  Lowest   │  Medium   │
                │  Priority │  Priority │
                └───────────┴───────────┘
```

### Trade-offs

| Approach | Pros | Cons |
|----------|------|------|
| **Simple model, few parameters** | Easy to calibrate, transparent | May miss important processes |
| **Complex model, many parameters** | More realistic | Harder to calibrate, more uncertainty |

The principle of **parsimony**: Use the simplest model that adequately represents the system.

### Connection to Our Model

From Notebook 6, recall that our model:
- Has only 3 observation wells
- Is unvalidated in most of the domain
- Uses steady-state assumption

The sensitivity analysis helps us understand:
- Which parameters most affect predictions in unobserved areas
- Where additional monitoring would be most valuable
- What uncertainty bounds we should place on predictions

## 6 - Advanced Topics

For masters-level work and professional practice, more sophisticated methods are available. This section provides a brief overview.

### Global Sensitivity Analysis

#### Morris Screening Method

An efficient method to identify the most important parameters from a large set.

- **How it works**: Computes "elementary effects" by varying one parameter at a time, but from different starting points in parameter space
- **Output**: Mean (μ) indicates overall sensitivity; standard deviation (σ) indicates interaction effects
- **When to use**: Initial screening before more expensive analysis
- **Tool**: `pestpp-sen` in PEST++ suite

#### Sobol Variance-Based Indices

Decomposes output variance into contributions from each parameter and their interactions.

- **First-order index (S₁)**: Variance due to parameter alone
- **Total-order index (Sᴛ)**: Variance due to parameter including all interactions
- **When to use**: When parameter interactions are important
- **Cost**: Requires many model runs (thousands)

### Monte Carlo Simulation

The workhorse of uncertainty quantification.

**Process**:
1. Define probability distributions for uncertain parameters
2. Randomly sample from distributions (100s-1000s of samples)
3. Run model for each sample
4. Analyze distribution of outputs

**Output**: Probability distributions of predictions, confidence intervals

### Latin Hypercube Sampling (LHS)

A more efficient alternative to pure random sampling.

- **How it works**: Divides each parameter range into equal-probability intervals and ensures each interval is sampled exactly once
- **Advantage**: Better coverage of parameter space with fewer samples
- **Typical use**: 50-500 samples instead of 1000s for pure Monte Carlo

### Bayesian Inference

A probabilistic framework for updating parameter estimates with data.

**Key concepts**:
- **Prior distribution**: What we believed before seeing data
- **Likelihood**: How well the model fits observed data
- **Posterior distribution**: Updated belief after seeing data

$$P(\theta|D) \propto P(D|\theta) \times P(\theta)$$

Where θ = parameters, D = data

**Tools**: PEST++ IES (Iterative Ensemble Smoother), pyEMU

### Data Worth Analysis

Quantifies the value of observations for reducing prediction uncertainty.

**Questions answered**:
- How much would a new observation well reduce uncertainty?
- Which existing observations are most valuable?
- Where should we collect additional data?

**Tool**: `pestpp-da` (Data Space Analysis) or pyEMU's `Schur` complement analysis

### Tools for Advanced Analysis

| Tool | Purpose | Access |
|------|---------|--------|
| **pyEMU** | Python interface to PEST++, sensitivity/uncertainty analysis for groundwater models | [GitHub](https://github.com/pypest/pyemu) |
| **PEST++ IES** | Iterative Ensemble Smoother for Bayesian calibration | [GitHub](https://github.com/usgs/pestpp) |
| **PEST++ SEN** | Morris method sensitivity analysis | Included in PEST++ |
| **SALib** | General-purpose Python library for sensitivity analysis (see below) | [GitHub](https://github.com/SALib/SALib) |

#### SALib: A Model-Independent Sensitivity Analysis Library

**SALib** (Sensitivity Analysis Library) is a Python library that implements several global sensitivity analysis methods. Unlike pyEMU/PEST++, which are designed specifically for environmental models, SALib is **model-independent** — it works with any model that can be called from Python.

**Key methods implemented**:
- Sobol sensitivity analysis
- Morris method (Elementary Effects)
- FAST (Fourier Amplitude Sensitivity Test)
- Delta Moment-Independent Measure
- Derivative-based Global Sensitivity Measures (DGSM)

**When to use SALib vs. pyEMU**:
- Use **SALib** when you want a lightweight, general-purpose library or when working with non-MODFLOW models
- Use **pyEMU/PEST++** when working with MODFLOW models, as they handle model files, observations, and parameters automatically

**Example workflow with SALib**:
1. Define parameter bounds and distributions
2. Generate parameter samples using SALib's sampling functions
3. Run your model for each sample (you write this loop)
4. Analyze results using SALib's analysis functions

> **Further Reading**: See Hill & Tiedeman (2007) "Effective Groundwater Model Calibration" for comprehensive coverage of these methods.

## 7 - Discussion

### Limitations of Our Analysis

The simple OAT sensitivity analysis performed above has limitations:

1. **Only two parameters tested**: Real models have many more uncertain parameters
2. **Single output metric**: Mean head doesn't capture spatial variability
3. **No parameter interactions**: OAT misses combined effects
4. **Steady-state only**: Transient systems may have different sensitivities

### When More Rigorous Methods Are Needed

Use advanced methods when:

- Decisions have significant consequences (regulatory, financial, environmental)
- Parameter interactions are expected
- You need quantitative uncertainty bounds
- Data worth analysis is needed to plan monitoring

### Connection to Your Case Study

> **In your group case study notebooks** (e.g., `student_work/group_0/case_study_transport_group_0.ipynb`), Section 15 guides you through sensitivity analysis for transport parameters:
>
> - Pumping rate sensitivity (±50%)
> - Advective transport parameters (K, porosity, gradient)
> - Impact on breakthrough times
>
> The concepts from this notebook provide the foundation for that practical work.

### Key Takeaways

1. **Sensitivity analysis** identifies which parameters most influence predictions
2. **Uncertainty analysis** quantifies the range of possible predictions
3. **OAT is a starting point** - use global methods for rigorous analysis
4. **Prioritize data collection** for sensitive, uncertain parameters
5. **Be honest** about what your analysis can and cannot tell you

## 8 - References

### Textbooks

- Anderson, M.P., Woessner, W.W., & Hunt, R.J. (2015). *Applied Groundwater Modeling* (2nd ed.). Academic Press. Chapter 11: Sensitivity Analysis. [ScienceDirect](https://www.sciencedirect.com/book/9780120581030/applied-groundwater-modeling)

- Hill, M.C., & Tiedeman, C.R. (2007). *Effective Groundwater Model Calibration: With Analysis of Data, Sensitivities, Predictions, and Uncertainty*. Wiley. [Wiley Online Library](https://onlinelibrary.wiley.com/doi/book/10.1002/0470041080)

### Software Documentation

- pyEMU Documentation: [https://github.com/pypest/pyemu](https://github.com/pypest/pyemu)

- PEST++ User Manual: [https://github.com/usgs/pestpp](https://github.com/usgs/pestpp)

- SALib (Sensitivity Analysis Library): [https://salib.readthedocs.io/](https://salib.readthedocs.io/)

### Key Papers

- Saltelli, A., et al. (2008). *Global Sensitivity Analysis: The Primer*. [Wiley](https://onlinelibrary.wiley.com/doi/book/10.1002/9780470725184).

- Morris, M.D. (1991). Factorial sampling plans for preliminary computational experiments. *Technometrics*, 33(2), 161-174. [JSTOR](https://www.jstor.org/stable/1269043)

- White, J.T. (2018). A model-independent iterative ensemble smoother for efficient history-matching and uncertainty quantification in very high dimensions. *Environmental Modelling & Software*, 109, 191-201. [ScienceDirect](https://doi.org/10.1016/j.envsoft.2018.06.009)